In [ ]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')

In [28]:
import pyarrow as pa
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.exceptions import NamespaceAlreadyExistsError
from pyiceberg.transforms import IdentityTransform, DayTransform

In [15]:
# 1. Connect to R2 Data Catalog

# Define catalog connection details

WAREHOUSE = os.environ['WAREHOUSE']
TOKEN = os.environ['TOKEN']
CATALOG_URI = os.environ['CATALOG_URI']

# Connect to R2 Data Catalog
catalog = RestCatalog(
    name="my_catalog",
    warehouse=WAREHOUSE,
    uri=CATALOG_URI,
    token=TOKEN,
)

# list namespaces
print(catalog.list_namespaces())

[('zczxczxc',)]


In [16]:
# List tables in the "default" namespace
tables = catalog.list_tables(namespace='zczxczxc')
print("Tables in 'default':", tables)

Tables in 'default': [('zczxczxc', 'fsdfsdfsdf')]


In [17]:
# Load a specific table
table = catalog.load_table(('zczxczxc', 'fsdfsdfsdf'))

# Print the schema
print(table.schema())

table {
  1: __ingest_ts: required timestamp
  2: user_id: required string
  3: event_name: optional string
}


In [18]:
# 1. Create namespace
catalog.create_namespace("default")

prueba(
  1: event_time: required timestamp,
  2: user_id: required string,
  3: event_data: optional string
),
partition by: [event_day],
sort order: [],
snapshot: null

In [27]:
# 2. Load the pipeline-created table
table = catalog.load_table("default.prueba")

print("Current Spec:", table.spec())

Current Spec: [
  1000: event_day: day(1)
  1001: user_id_partition: identity(2)
]


In [26]:
# verifications
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("default.prueba")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # remove and then re add as last partition
    update.remove_field("timestamp_day") 
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")
    update.add_field("timestamp", DayTransform(), "date")

print("New Spec:", table.spec())

Partition spec updated successfully!
[
  1000: event_day: day(1)
  1001: user_id_partition: identity(2)
]


In [21]:
# List tables in the "default" namespace
new_tables = catalog.list_tables(namespace='default')
print("Tables in 'default':", new_tables)

Tables in 'default': [('default', 'prueba')]


In [22]:
# Load a specific table
table = catalog.load_table(('default', 'prueba'))

# Print the schema
print(table.schema())

table {
  1: event_time: required timestamp
  2: user_id: required string
  3: event_data: optional string
}
